# Security exploit attempts: handled vs not handled

Local-only notebook showing exploit-attempt behavior for DataGateway, DataHelper, and Datacube-backed loads.

In [1]:
from __future__ import annotations

import tempfile
from pathlib import Path

import pandas as pd
from sqlalchemy import create_engine, text

from boti_data import DataGateway, DataHelper, DatacubeConfig, DatacubeContract, SqlDatabaseConfig

tmp_dir = tempfile.TemporaryDirectory()
db_path = Path(tmp_dir.name) / "security_demo.db"

engine = create_engine(f"sqlite:///{db_path}")
with engine.begin() as conn:
    conn.execute(text("CREATE TABLE users (id INTEGER PRIMARY KEY, status TEXT)"))
    conn.execute(text("INSERT INTO users (status) VALUES ('active'), ('inactive')"))
engine.dispose()

cfg = SqlDatabaseConfig(
    connection_url=f"sqlite:///{db_path}",
    poolclass="sqlalchemy.pool.NullPool",
    query_only=False,
)

def user_count() -> int:
    eng = create_engine(f"sqlite:///{db_path}")
    try:
        with eng.connect() as conn:
            return int(conn.execute(text("SELECT COUNT(*) FROM users")).scalar_one())
    finally:
        eng.dispose()

rows = []
def rec(component, case, expected, observed, handled, notes):
    rows.append({"component": component, "case": case, "expected": expected, "observed": observed, "handled": handled, "notes": notes})

In [2]:
# DataGateway: blocked mutating SQL
try:
    with DataGateway(cfg) as gw:
        gw.load(sql="DELETE FROM users", as_pandas=True, allow_raw_sql=True)
    rec("DataGateway", "mutating raw sql", "blocked", "allowed", False, "Unexpected")
except ValueError as exc:
    rec("DataGateway", "mutating raw sql", "blocked", "blocked", True, str(exc))

# DataGateway: handled parameterized payload
payload = "active'; DROP TABLE users; --"
with DataGateway(cfg) as gw:
    frame = gw.load(sql="SELECT id, status FROM users WHERE status = :status", params={"status": payload}, as_pandas=True, allow_raw_sql=True)
ok = frame.empty and user_count() == 2
rec("DataGateway", "parameterized payload", "handled", "handled" if ok else "not handled", bool(ok), "Bound params avoid SQL injection execution.")

# DataGateway: not handled when application interpolates unsafely
tautology = "active' OR 1=1 --"
unsafe_sql = f"SELECT id, status FROM users WHERE status = '{tautology}'"
with DataGateway(cfg) as gw:
    unsafe = gw.load(sql=unsafe_sql, as_pandas=True, allow_raw_sql=True)
vulnerable = len(unsafe) > 1
rec("DataGateway", "unsafe interpolation", "blocked/neutralized", "not handled" if vulnerable else "handled", not vulnerable, "App must avoid string interpolation.")

In [3]:
# DataHelper inherits the same policy and limits
try:
    with DataHelper(cfg, raw_sql_policy="disabled") as helper:
        helper.load(sql="SELECT 1 AS id", as_pandas=True, allow_raw_sql=True)
    rec("DataHelper", "policy disabled", "blocked", "allowed", False, "Unexpected")
except ValueError as exc:
    rec("DataHelper", "policy disabled", "blocked", "blocked", True, str(exc))

# Datacube without validator: not handled by default
def loader(req):
    return pd.DataFrame([{"cube": req.cube, "filters": sorted([str(k) for k in req.filters.keys()])}])

with DataGateway(DatacubeConfig(loader=loader, default_cube="sales")) as cube_gw:
    f = cube_gw.load(cube="../admin", filters={"$where": "1=1"}, return_type="pandas")
accepted = not f.empty
rec("Datacube", "no validator", "blocked", "not handled" if accepted else "blocked", not accepted, "Add request_validator for policy enforcement.")

# Datacube with validator: handled
def validator(req):
    if ".." in str(req.cube or ""):
        raise ValueError("invalid cube")
    if any(str(k).startswith("$") for k in req.filters):
        raise ValueError("invalid filter key")

contract = DatacubeContract(request_validator=validator)
try:
    with DataGateway(DatacubeConfig(loader=loader, contract=contract, default_cube="sales")) as cube_gw:
        cube_gw.load(cube="../admin", filters={"$where": "1=1"}, return_type="pandas")
    rec("Datacube", "with validator", "blocked", "allowed", False, "Unexpected")
except ValueError as exc:
    rec("Datacube", "with validator", "blocked", "blocked", True, str(exc))

In [4]:
report = pd.DataFrame(rows)[["component", "case", "expected", "observed", "handled", "notes"]]
report

,component,case,expected,observed,handled,notes
0,DataGateway,mutating raw sql,blocked,blocked,True,1 validation error for SqlLoadRequest\n Value...
1,DataGateway,parameterized payload,handled,handled,True,Bound params avoid SQL injection execution.
2,DataGateway,unsafe interpolation,blocked/neutralized,not handled,False,App must avoid string interpolation.
3,DataHelper,policy disabled,blocked,blocked,True,Raw sql= execution is disabled by this DataGat...
4,Datacube,no validator,blocked,not handled,False,Add request_validator for policy enforcement.
5,Datacube,with validator,blocked,blocked,True,Datacube contract request validation failed: i...


In [5]:
tmp_dir.cleanup()
print("Cleaned up temp resources.")

Cleaned up temp resources.
